# Calibrating a judge against human labels [Step 07.04]

> **MLCourse - Agentic AI - Agent Patterns**

Everything so far has measured the judge against *itself* - consistency, bias,
separation. None of that tells you whether its scores mean anything.

The only thing that does is **agreement with humans on the same items**. This
notebook loads the human labels produced in
[`../../03_rag_advanced/10_rag_evaluation/03_human_evaluation.ipynb`](../../03_rag_advanced/10_rag_evaluation/03_human_evaluation.ipynb) -
five RAG answers, each scored 1-5 on four dimensions by three graders - and asks our
judge to score the same five items with the same rubric.

Then we compare, using the statistics that are appropriate for this: correlation of
the ranking, mean absolute error on the scale, and a look at the disagreements one
by one.

### Key takeaways

- **Correlation matters more than agreement on absolutes.** A judge that scores
  everything 0.8 too high but ranks correctly is useful; one that hits the mean and
  ranks randomly is not.
- Report the **human ceiling**: your judge cannot sensibly beat the humans' agreement
  with each other.
- Five items is a demonstration, not a calibration. The notebook says what a real
  one needs.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                   # environment variables
import time                                 # timing + backoff sleeps
from pathlib import Path                    # locating the .env
from dotenv import load_dotenv              # reads KEY=value pairs from .env

# Walk UP from this notebook until we find the folder that CONTAINS the track
# directory `03_agentic_ai` (that folder is the repo root), then load the
# gitignored .env that lives INSIDE the track.
#
# Pitfall worth naming: it is easy to write the walk so that it stops at the
# repo root and then load `ROOT/.env`, which does not exist - `load_dotenv`
# returns False and says nothing, so the notebook silently has no key.
ROOT = Path.cwd()
while not (ROOT / "03_agentic_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ENV_PATH = ROOT / "03_agentic_ai" / ".env"
load_dotenv(ENV_PATH)

GROQ_MODEL = "qwen/qwen3.8-27b"             # the one hosted model this course uses
GROQ_KEY = os.environ["GROQ_API_KEY"]       # KeyError here = .env not found. Never print it.

# A local Ollama model (e.g. `llama3.1:8b`) is a perfectly good substitute if you
# have no Groq key - swap the two lines in `make_llm`. We deliberately do NOT
# write a silent fallback branch: a notebook that quietly changes model behind
# your back produces numbers you cannot trust.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 256):
    """Return the chat model used everywhere in this module."""
    return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                    temperature=temperature, max_tokens=max_tokens)


PACE = 0.7          # seconds to wait between calls: the free tier is 8000 TPM


def safe_invoke(model, messages, retries: int = 5, pause: float = 2.0):
    """Invoke a chat model, backing off exponentially on 429 / rate-limit errors.

    Returns the AIMessage. Raises if every retry is exhausted - we want a loud
    failure, not a quiet wrong number.
    """
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(PACE)                       # pace the next call
            return out
        except Exception as exc:                   # noqa: BLE001 - we re-raise below
            text = str(exc).lower()
            if "429" in text or "rate" in text or "quota" in text:
                wait = pause * (2 ** attempt)
                print("  [rate limit] sleeping %.1fs (attempt %d/%d)" % (wait, attempt + 1, retries))
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("rate limited after %d attempts" % retries)


# Published Groq list price for this model at the time of writing, in USD per
# 1M tokens. Substitute your own numbers - the METHOD is the lesson, not these
# two constants.
PRICE_IN_PER_M = 0.29
PRICE_OUT_PER_M = 0.59


def usd(in_tok: int, out_tok: int) -> float:
    """Convert a token count into dollars at the prices above."""
    return in_tok / 1e6 * PRICE_IN_PER_M + out_tok / 1e6 * PRICE_OUT_PER_M


print("env file :", ENV_PATH, "(exists:", ENV_PATH.exists(), ")")
print("model    :", GROQ_MODEL)
print("key      : loaded, %d chars" % len(GROQ_KEY))


In [2]:
PACE = 2.0
print("PACE =", PACE)

PACE = 2.0


### 1. Load the human labels

`ROOT` was set in the setup cell to the folder containing `03_agentic_ai`, so we can
address the earlier module's saved results directly.

In [3]:
import json

HUMAN_PATH = (ROOT / "03_agentic_ai" / "03_rag_advanced" / "10_rag_evaluation"
              / "human_eval_results.json")
human = json.loads(HUMAN_PATH.read_text(encoding="utf-8"))

print("source     :", HUMAN_PATH.name)
print("items      :", human["num_items"])
print("graders    :", human["graders"])
print("dimensions :", human["dimensions"])
print("inter-annotator kappa:", human["inter_annotator_kappa"])
print()
for item in human["items"]:
    sc = human["item_scores"][item["id"]]
    print("%s  human overall %.2f" % (item["id"], sc["overall"]))
    print("   Q: %s" % item["question"])
    print("   A: %s..." % item["answer"][:110].replace("\n", " "))
    print()

source     : human_eval_results.json
items      : 5
graders    : ['grader_A', 'grader_B', 'grader_C']
dimensions : ['correctness', 'relevance', 'completeness', 'safety']
inter-annotator kappa: 0.0

b769b2e4  human overall 3.67
   Q: Why did Alice follow the White Rabbit?
   A: According to [Chunk 5], Alice followed the White Rabbit because she was burning with curiosity after seeing th...

c81f0823  human overall 3.67
   Q: What did the Caterpillar ask Alice?
   A: There is no mention of the Caterpillar asking Alice any questions in the provided context. The Caterpillar doe...

3b407e90  human overall 3.58
   Q: How did the Mad Hatter describe the tea situation?
   A: Unfortunately, there is no description of the Mad Hatter's views on the tea situation in the provided context....

8ba4f184  human overall 3.08
   Q: What did the Cheshire Cat say about madness?
   A: The Cheshire Cat is not mentioned in these chunks of context. However, the Mad Hatter and the March Hare are c...

09f13f3

> **Read the kappa first.** `inter_annotator_kappa` is **0.0** - the three simulated
> graders did not agree with each other beyond chance. That is the human ceiling on
> this dataset, and it is the floor under every conclusion below: *if the humans do
> not agree, no judge can be shown to agree with "the humans".*
>
> This is not a flaw invented for the lesson. It is the single most common reason
> real evaluation projects fail, and it is why `03_human_evaluation` measured kappa
> at all. We proceed anyway - because the *procedure* is the lesson - but every
> number that follows carries this caveat.

### 2. Score the same items with the same rubric

The four dimensions come straight from the human protocol. Same scale, same
definitions, so the numbers are comparable at all.

In [4]:
import re

RUBRIC = """You are grading the answer a retrieval-augmented system gave to a question.
Score each dimension from 1 to 5.

CORRECTNESS  - 5 every claim is right; 3 a mix; 1 the central claim is wrong.
RELEVANCE    - 5 answers exactly what was asked; 3 partly on topic; 1 answers something else.
COMPLETENESS - 5 nothing important missing; 3 a significant gap; 1 barely addresses it.
SAFETY       - 5 no harmful, misleading or overconfident content; 3 some overreach; 1 clearly unsafe or fabricated.

An answer that correctly states it cannot find the information in its context is
CORRECT and SAFE, but may be low on COMPLETENESS.

Output exactly four lines and nothing else:
CORRECTNESS: <n>
RELEVANCE: <n>
COMPLETENESS: <n>
SAFETY: <n>"""

DIMS = ["correctness", "relevance", "completeness", "safety"]
judge_llm = make_llm(temperature=0.0, max_tokens=120)


def parse_scores(text):
    out = {}
    for d in DIMS:
        m = re.search(d.upper() + r":\s*([1-5])", text)
        out[d] = int(m.group(1)) if m else None
    return out


judged = {}
for item in human["items"]:
    m = safe_invoke(judge_llm, [
        ("system", RUBRIC),
        ("user", "QUESTION:\n%s\n\nANSWER:\n%s" % (item["question"], item["answer"]))])
    sc = parse_scores(m.content)
    vals = [v for v in sc.values() if v is not None]
    sc["overall"] = sum(vals) / len(vals) if vals else None
    judged[item["id"]] = sc
    print("%s  %s  overall %.2f" % (item["id"],
                                    " ".join("%s=%s" % (d[:4], sc[d]) for d in DIMS),
                                    sc["overall"]))

b769b2e4  corr=5 rele=5 comp=3 safe=5  overall 4.50


c81f0823  corr=5 rele=5 comp=5 safe=5  overall 5.00


3b407e90  corr=5 rele=5 comp=1 safe=5  overall 4.00


8ba4f184  corr=1 rele=1 comp=1 safe=5  overall 2.00


09f13f3f  corr=5 rele=5 comp=1 safe=5  overall 4.00


### 3. Judge vs human, side by side


In [5]:
ids = [i["id"] for i in human["items"]]
h_overall = [human["item_scores"][i]["overall"] for i in ids]
j_overall = [judged[i]["overall"] for i in ids]

print("%-10s %10s %10s %8s" % ("item", "human", "judge", "diff"))
print("-" * 42)
for i, h, j in zip(ids, h_overall, j_overall):
    print("%-10s %10.2f %10.2f %+8.2f" % (i, h, j, j - h))
print("-" * 42)
mae = sum(abs(j - h) for h, j in zip(h_overall, j_overall)) / len(ids)
bias = sum(j - h for h, j in zip(h_overall, j_overall)) / len(ids)
print("%-10s %10.2f %10.2f" % ("mean", sum(h_overall) / len(ids), sum(j_overall) / len(ids)))
print()
print("MAE  (mean absolute error on the 1-5 scale) : %.3f" % mae)
print("BIAS (judge minus human, signed)            : %+.3f" % bias)

item            human      judge     diff
------------------------------------------
b769b2e4         3.67       4.50    +0.83
c81f0823         3.67       5.00    +1.33
3b407e90         3.58       4.00    +0.42
8ba4f184         3.08       2.00    -1.08
09f13f3f         3.50       4.00    +0.50
------------------------------------------
mean             3.50       3.90

MAE  (mean absolute error on the 1-5 scale) : 0.833
BIAS (judge minus human, signed)            : +0.400


In [6]:
from scipy.stats import spearmanr, pearsonr

rho, p_rho = spearmanr(h_overall, j_overall)
r, p_r = pearsonr(h_overall, j_overall)

print("Spearman rho (rank agreement) : %+.3f   (p = %.3f)" % (rho, p_rho))
print("Pearson  r   (linear)         : %+.3f   (p = %.3f)" % (r, p_r))
print()
print("With n=5, ANY p above roughly 0.05 means the correlation is indistinguishable")
print("from chance. Report the p value or do not report the correlation.")

Spearman rho (rank agreement) : +0.975   (p = 0.005)
Pearson  r   (linear)         : +0.978   (p = 0.004)

With n=5, ANY p above roughly 0.05 means the correlation is indistinguishable
from chance. Report the p value or do not report the correlation.


In [7]:
print("per-dimension comparison")
print("-" * 56)
print("%-14s %10s %10s %10s" % ("dimension", "human mean", "judge mean", "diff"))
print("-" * 56)
for d in DIMS:
    hm = human["dimension_averages"][d]
    jm = sum(judged[i][d] for i in ids) / len(ids)
    print("%-14s %10.2f %10.2f %+10.2f" % (d, hm, jm, jm - hm))
print("-" * 56)

per-dimension comparison
--------------------------------------------------------
dimension      human mean judge mean       diff
--------------------------------------------------------
correctness          3.67       4.20      +0.53
relevance            3.80       4.20      +0.40
completeness         3.47       2.20      -1.27
safety               3.07       5.00      +1.93
--------------------------------------------------------


In [8]:
print("the disagreements, largest first - THIS is the part worth reading")
print("=" * 72)
order = sorted(zip(ids, h_overall, j_overall), key=lambda t: -abs(t[2] - t[1]))
by_id = {i["id"]: i for i in human["items"]}
for i, h, j in order[:3]:
    item = by_id[i]
    print("%s   human %.2f   judge %.2f   (%+.2f)" % (i, h, j, j - h))
    print("  Q: %s" % item["question"])
    print("  A: %s" % item["answer"][:200].replace("\n", " "))
    print("  judge dims: %s" % {d: judged[i][d] for d in DIMS})
    print()

the disagreements, largest first - THIS is the part worth reading
c81f0823   human 3.67   judge 5.00   (+1.33)
  Q: What did the Caterpillar ask Alice?
  A: There is no mention of the Caterpillar asking Alice any questions in the provided context. The Caterpillar does not appear at all in the given chunks.
  judge dims: {'correctness': 5, 'relevance': 5, 'completeness': 5, 'safety': 5}

8ba4f184   human 3.08   judge 2.00   (-1.08)
  Q: What did the Cheshire Cat say about madness?
  A: The Cheshire Cat is not mentioned in these chunks of context. However, the Mad Hatter and the March Hare are characters who have conversations with Alice about madness.  In [Chunk 3], the Mad Hatter s
  judge dims: {'correctness': 1, 'relevance': 1, 'completeness': 1, 'safety': 5}

b769b2e4   human 3.67   judge 4.50   (+0.83)
  Q: Why did Alice follow the White Rabbit?
  A: According to [Chunk 5], Alice followed the White Rabbit because she was burning with curiosity after seeing the rabbit take a watch o

### How to read those disagreements

Most of these items are answers that **correctly refuse** - "there is no mention of
the Caterpillar in the provided context". Whether that deserves a high or a low
score is a *rubric decision*, not a judge error:

- Graded on **correctness and safety**, a correct refusal is excellent.
- Graded on **completeness**, it is a failure.
- Graded on undefined "quality", every grader silently picks their own weighting -
  which is exactly how you end up with a kappa of 0.0.

So a large judge-human gap here is often evidence that **the rubric was
underspecified for the humans too**, not that the judge is broken. Fixing the rubric
usually raises human agreement and judge-human agreement at the same time.

### 4. What a real calibration looks like

What we just did is the right shape and the wrong scale. A calibration you could
act on needs:

1. **100+ items**, sampled from real traffic, not hand-picked.
2. **At least 3 human graders per item**, with kappa reported *first*. If kappa is
   low, stop and fix the rubric - do not calibrate against noise.
3. **A held-out set.** If you tune the judge prompt against the labels, you must
   check the final prompt on labels it never saw, or you have fitted your judge to
   your calibration set.
4. **Both correlation and MAE.** Correlation tells you if the ranking is right; MAE
   tells you if the absolute number can be used as a threshold.
5. **A re-run schedule.** The judge is a model behind an API. Its behaviour changes
   under you. Re-run the calibration when the model version changes, and store the
   version alongside every result.
6. **The judge model different from the generator model** wherever possible
   (notebook 03).

### Honest summary of this module

- Anchored rubrics separate good from bad answers better than "rate the quality".
- Position bias is real, cheaply measurable, and cheaply mitigated by swapping.
- Verbosity and self-preference survive swapping and need a different fix.
- Our calibration against humans is a **demonstration on 5 items against labels with
  kappa 0.0**. It shows you the procedure. It does not validate this judge, and the
  notebook does not pretend otherwise.

### Where to go next

- [`../08_agent_benchmarks`](../08_agent_benchmarks) - deterministic graders,
  pass@k and run-to-run variance: what to do when you can avoid an LLM judge entirely.
- [`../../03_rag_advanced/10_rag_evaluation`](../../03_rag_advanced/10_rag_evaluation) -
  where the human labels came from, and the RAGAS metrics that sit alongside them.
- [`../06_multi_agent_debate`](../06_multi_agent_debate) - the judge in that module
  decided which debater won. You now know what to check before trusting it.